# Tech COE Allocation Summary — Gold Analytics

Leadership-facing metrics built on `arganoadw_oacuser_sh.oacuser.TECH_COE_ALLOCATIONS` (the bronze fact: scheduled hours per **Arganaut × project × week**).

**Capacity assumption:** `40` hrs/week per Arganaut (edit `CAPACITY_HOURS_PER_WEEK`).

**Billable vs non-billable:** every scheduled hour is a project allocation, so **billable hours = scheduled hours** and **non-billable hours = 40 − billable** (the unallocated capacity remainder, i.e. bench). We do not track true billable/non-billable here — these are forecasted allocations.

## Gold tables produced
| Table | Grain | Use |
|---|---|---|
| `TECH_COE_ALLOC_DETAIL` | dept × arganaut × project × week | Flexible base — **slice by Department and Project** |
| `TECH_COE_ALLOC_UTILIZATION_PERSON_WEEK` | arganaut × week | Per-person utilization, over-allocation, bench |
| `TECH_COE_ALLOC_DEPT_WEEK` | dept × week | Utilization %, billable (scheduled) vs non-billable (bench) hours, over-allocated count |
| `TECH_COE_ALLOC_PROJECT_WEEK` | project × week | Project demand / staffing over time |
| `TECH_COE_ALLOC_PROJECT_RISK` | project | Total hours, headcount, key-person-risk flag |
| `TECH_COE_ALLOC_DEPT_FORECAST` | dept × week (+ future) | Trend + moving-average demand projection |

**Forecast:** linear trend + 3-week moving average projected `FORECAST_WEEKS` ahead per department. Directional planning aid (booking pace), not a statistical model — allocations naturally taper in future weeks as the pipeline fills (the "coverage cliff").

In [ ]:
# ============================================================
# TECH COE ALLOCATION SUMMARY — Gold Analytics
# Builds leadership metrics + a demand forecast over the
# bronze TECH_COE_ALLOCATIONS fact. Full overwrite each run.
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, DoubleType, IntegerType
)
import pandas as pd
import numpy as np

spark = SparkSession.builder.appName('tech_coe_allocations_gold').getOrCreate()

# ─── CONFIG ────────────────────────────────────────
SOURCE_TABLE  = 'arganoadw_oacuser_sh.oacuser.TECH_COE_ALLOCATIONS'
TARGET_SCHEMA = 'arganoadw_oacuser_sh.oacuser'

CAPACITY_HOURS_PER_WEEK = 40.0   # full-time weekly capacity per Arganaut
FORECAST_WEEKS          = 8      # weeks to project beyond the last scheduled week
# Billable = scheduled (allocated) hours.  Non-billable = CAP - billable (bench).
# ───────────────────────────────────────────────

CAP = CAPACITY_HOURS_PER_WEEK

def gold_table(name):
    return f'{TARGET_SCHEMA}.{name}'

def write_gold(df, name):
    # External catalog rules (same as bronze): no Delta format, no CREATE SCHEMA,
    # overwrite + overwriteSchema via saveAsTable with the 3-part name.
    tbl = gold_table(name)
    (df.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(tbl))
    print(f'✅ wrote {tbl}  ({df.count():,} rows)')
    return tbl

In [ ]:
# ── Load bronze fact ──────────────────────────────
base = spark.table(SOURCE_TABLE)
total = base.count()
base.createOrReplaceTempView('alloc')

rng = base.selectExpr('min(WEEK_START_DATE) AS lo', 'max(WEEK_START_DATE) AS hi').first()
print(f'Source rows: {total:,}')
print(f'Week range:  {rng.lo} → {rng.hi}')

In [ ]:
# ── Person-week utilization ──────────────────────────
#   billable (scheduled) vs non-billable (CAP - scheduled, floored at 0).
person_week = spark.sql(f'''
    SELECT DEPARTMENT, ARGANAUT, YEAR_MONTH, WEEK_START_DATE,
           ROUND(SUM(SCHEDULED_HOURS), 2)                          AS SCHEDULED_HOURS,
           {CAP}                                                   AS CAPACITY_HOURS,
           ROUND(SUM(SCHEDULED_HOURS) / {CAP} * 100, 1)            AS UTILIZATION_PCT,
           ROUND(SUM(SCHEDULED_HOURS), 2)                          AS BILLABLE_HOURS,
           ROUND(GREATEST({CAP} - SUM(SCHEDULED_HOURS), 0), 2)     AS NONBILLABLE_HOURS,
           ROUND({CAP} - SUM(SCHEDULED_HOURS), 2)                  AS AVAILABLE_HOURS,
           CASE WHEN SUM(SCHEDULED_HOURS) > {CAP} THEN 1 ELSE 0 END AS IS_OVER_ALLOCATED,
           CASE WHEN SUM(SCHEDULED_HOURS) = 0    THEN 1 ELSE 0 END  AS IS_ON_BENCH
    FROM alloc
    GROUP BY DEPARTMENT, ARGANAUT, YEAR_MONTH, WEEK_START_DATE
''')
person_week.createOrReplaceTempView('person_week')
write_gold(person_week, 'TECH_COE_ALLOC_UTILIZATION_PERSON_WEEK')

In [ ]:
# ── Department-week summary (the exec headline) ──────────
#   BILLABLE_HOURS    = scheduled (allocated) hours
#   NONBILLABLE_HOURS = capacity - scheduled = bench (BENCH_HOURS alias)
dept_week = spark.sql(f'''
    SELECT DEPARTMENT, YEAR_MONTH, WEEK_START_DATE,
           COUNT(DISTINCT ARGANAUT)                                        AS ROSTER_HEADCOUNT,
           COUNT(DISTINCT CASE WHEN SCHEDULED_HOURS > 0 THEN ARGANAUT END) AS ALLOCATED_HEADCOUNT,
           ROUND(SUM(SCHEDULED_HOURS), 2)                                  AS TOTAL_SCHEDULED_HOURS,
           ROUND(SUM(SCHEDULED_HOURS), 2)                                  AS BILLABLE_HOURS,
           ROUND(COUNT(DISTINCT ARGANAUT) * {CAP}, 2)                      AS TOTAL_CAPACITY_HOURS,
           ROUND(SUM(SCHEDULED_HOURS) / (COUNT(DISTINCT ARGANAUT) * {CAP}) * 100, 1) AS UTILIZATION_PCT,
           ROUND(GREATEST(COUNT(DISTINCT ARGANAUT) * {CAP} - SUM(SCHEDULED_HOURS), 0), 2) AS NONBILLABLE_HOURS,
           ROUND(COUNT(DISTINCT ARGANAUT) * {CAP} - SUM(SCHEDULED_HOURS), 2) AS BENCH_HOURS
    FROM alloc
    GROUP BY DEPARTMENT, YEAR_MONTH, WEEK_START_DATE
''')

pw_agg = spark.sql('''
    SELECT DEPARTMENT, YEAR_MONTH, WEEK_START_DATE,
           SUM(IS_OVER_ALLOCATED) AS OVER_ALLOCATED_COUNT,
           SUM(IS_ON_BENCH)       AS BENCH_HEADCOUNT
    FROM person_week
    GROUP BY DEPARTMENT, YEAR_MONTH, WEEK_START_DATE
''')

dept_week = dept_week.join(pw_agg, ['DEPARTMENT', 'YEAR_MONTH', 'WEEK_START_DATE'], 'left')
dept_week.createOrReplaceTempView('dept_week')
write_gold(dept_week, 'TECH_COE_ALLOC_DEPT_WEEK')

In [ ]:
# ── Project demand (per week) + project risk (totals) ────
project_week = spark.sql('''
    SELECT DEPARTMENT, PROJECT_CODE, PROJECT_NAME,
           YEAR_MONTH, WEEK_START_DATE,
           ROUND(SUM(SCHEDULED_HOURS), 2)                                 AS SCHEDULED_HOURS,
           COUNT(DISTINCT CASE WHEN SCHEDULED_HOURS > 0 THEN ARGANAUT END) AS HEADCOUNT
    FROM alloc
    GROUP BY DEPARTMENT, PROJECT_CODE, PROJECT_NAME, YEAR_MONTH, WEEK_START_DATE
''')
write_gold(project_week, 'TECH_COE_ALLOC_PROJECT_WEEK')

project_risk = spark.sql('''
    SELECT PROJECT_CODE, PROJECT_NAME,
           ROUND(SUM(SCHEDULED_HOURS), 2)                                 AS TOTAL_SCHEDULED_HOURS,
           COUNT(DISTINCT CASE WHEN SCHEDULED_HOURS > 0 THEN ARGANAUT END) AS DISTINCT_PEOPLE,
           COUNT(DISTINCT CASE WHEN SCHEDULED_HOURS > 0 THEN WEEK_START_DATE END) AS ACTIVE_WEEKS,
           CASE WHEN COUNT(DISTINCT CASE WHEN SCHEDULED_HOURS > 0 THEN ARGANAUT END) = 1
                THEN 1 ELSE 0 END                                         AS IS_KEY_PERSON_RISK
    FROM alloc
    GROUP BY PROJECT_CODE, PROJECT_NAME
    HAVING SUM(SCHEDULED_HOURS) > 0
''')
write_gold(project_risk, 'TECH_COE_ALLOC_PROJECT_RISK')

In [ ]:
# ── Detail fact — flexible base for slicing by Dept & Project ──
detail = spark.sql('''
    SELECT DEPARTMENT, ARGANAUT, PROJECT_CODE, PROJECT_NAME,
           YEAR_MONTH, WEEK_START_DATE, SCHEDULED_HOURS
    FROM alloc
''')
write_gold(detail, 'TECH_COE_ALLOC_DETAIL')

In [ ]:
# ── Demand forecast per department ────────────────────
#   linear least-squares trend + 3-week moving average,
#   projected FORECAST_WEEKS past the last scheduled week.
#   Directional booking-pace aid, not a statistical model.
from datetime import timedelta

pdf = dept_week.select('DEPARTMENT', 'WEEK_START_DATE', 'TOTAL_SCHEDULED_HOURS').toPandas()
pdf['WEEK_START_DATE'] = pd.to_datetime(pdf['WEEK_START_DATE'])

rows = []
for dept, g in pdf.groupby('DEPARTMENT'):
    g = g.sort_values('WEEK_START_DATE').reset_index(drop=True)
    ma = g['TOTAL_SCHEDULED_HOURS'].rolling(3, min_periods=1).mean()
    x = ((g['WEEK_START_DATE'] - g['WEEK_START_DATE'].min()).dt.days / 7.0).values
    y = g['TOTAL_SCHEDULED_HOURS'].astype(float).values
    if len(g) >= 2:
        slope, intercept = np.polyfit(x, y, 1)
    else:
        slope, intercept = 0.0, (y[0] if len(y) else 0.0)
    for i in range(len(g)):
        rows.append([dept, g['WEEK_START_DATE'][i].date(), round(float(y[i]), 2),
                     round(float(intercept + slope * x[i]), 2), round(float(ma[i]), 2), 0])
    last_week = g['WEEK_START_DATE'].max()
    last_x = float(x.max()) if len(x) else 0.0
    last_ma = float(g['TOTAL_SCHEDULED_HOURS'].tail(3).mean())
    for k in range(1, FORECAST_WEEKS + 1):
        fdate = (last_week + timedelta(weeks=k)).date()
        rows.append([dept, fdate, None,
                     round(float(intercept + slope * (last_x + k)), 2),
                     round(last_ma, 2), 1])

fc = pd.DataFrame(rows, columns=['DEPARTMENT', 'WEEK_START_DATE',
                                 'ACTUAL_SCHEDULED_HOURS', 'TREND_HOURS',
                                 'MA_3WK_HOURS', 'IS_FORECAST'])
fc['WEEK_START_DATE'] = pd.to_datetime(fc['WEEK_START_DATE'])
for c in ['ACTUAL_SCHEDULED_HOURS', 'TREND_HOURS', 'MA_3WK_HOURS']:
    fc[c] = fc[c].astype(float)
fc['TREND_HOURS'] = fc['TREND_HOURS'].clip(lower=0)
fc['IS_FORECAST'] = fc['IS_FORECAST'].astype(int)

fc_schema = StructType([
    StructField('DEPARTMENT',             StringType(),  True),
    StructField('WEEK_START_DATE',        DateType(),    True),
    StructField('ACTUAL_SCHEDULED_HOURS', DoubleType(),  True),
    StructField('TREND_HOURS',            DoubleType(),  True),
    StructField('MA_3WK_HOURS',           DoubleType(),  True),
    StructField('IS_FORECAST',            IntegerType(), True),
])
forecast = spark.createDataFrame(fc, schema=fc_schema)
write_gold(forecast, 'TECH_COE_ALLOC_DEPT_FORECAST')

In [ ]:
# ── Verify / preview ──────────────────────────────
print('── Utilization % by department, most recent weeks ──')
spark.sql(f'''
    SELECT DEPARTMENT, WEEK_START_DATE, ROSTER_HEADCOUNT,
           UTILIZATION_PCT, BILLABLE_HOURS, NONBILLABLE_HOURS,
           OVER_ALLOCATED_COUNT
    FROM {gold_table('TECH_COE_ALLOC_DEPT_WEEK')}
    ORDER BY WEEK_START_DATE DESC, DEPARTMENT
    LIMIT 40
''').show(40, truncate=False)

print('── Coverage cliff: company-wide utilization by week ──')
spark.sql(f'''
    SELECT WEEK_START_DATE,
           ROUND(SUM(TOTAL_SCHEDULED_HOURS) / SUM(TOTAL_CAPACITY_HOURS) * 100, 1) AS UTILIZATION_PCT,
           ROUND(SUM(NONBILLABLE_HOURS), 0) AS NONBILLABLE_HOURS
    FROM {gold_table('TECH_COE_ALLOC_DEPT_WEEK')}
    GROUP BY WEEK_START_DATE
    ORDER BY WEEK_START_DATE
''').show(60, truncate=False)

print('── Top 15 projects by scheduled hours (key-person risk flagged) ──')
spark.sql(f'''
    SELECT PROJECT_CODE, PROJECT_NAME,
           TOTAL_SCHEDULED_HOURS, DISTINCT_PEOPLE, IS_KEY_PERSON_RISK
    FROM {gold_table('TECH_COE_ALLOC_PROJECT_RISK')}
    ORDER BY TOTAL_SCHEDULED_HOURS DESC
    LIMIT 15
''').show(15, truncate=False)

print('── Forecast (future weeks) ──')
spark.sql(f'''
    SELECT DEPARTMENT, WEEK_START_DATE, TREND_HOURS, MA_3WK_HOURS
    FROM {gold_table('TECH_COE_ALLOC_DEPT_FORECAST')}
    WHERE IS_FORECAST = 1
    ORDER BY DEPARTMENT, WEEK_START_DATE
    LIMIT 40
''').show(40, truncate=False)